[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/refactor/examples/benchmark.ipynb)

# MISDA benchmark

This notebook runs every canonical and synthetic MOP case separately. For each case, it prints the problem declaration, executes the public `static` analysis, displays `result.report()`, and finally draws the structural graph with `result.graph_plot()`.

In [ ]:
# Install MISDA from the refactor branch and make the example generators available.
import os
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/monacofj/misda.git"
BRANCH = "refactor"

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples" / "benchmarks" / "cases.py").is_file():
            return candidate
    return None

repository_root = find_repository_root()
if repository_root is None:
    repository_root = Path.cwd() / "misda-refactor"
    if repository_root.exists():
        subprocess.run(["git", "-C", str(repository_root), "switch", BRANCH], check=True)
        subprocess.run(["git", "-C", str(repository_root), "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(repository_root)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", f"git+{REPOSITORY_URL}@{BRANCH}"],
    check=True,
)
os.chdir(repository_root)
print(f"MISDA installed from branch '{BRANCH}'. Repository: {repository_root}")

In [ ]:
import misda

from examples.benchmarks.cases import CANONICAL_CASES
from examples.mop_definitions import (
    mopA_monotonic_redundancy,
    mopB_tradeoff_with_redundancies,
    mopC_latent_blocks_4x5,
    mopD_pure_conflict_groups,
    mopE_partial_redundancy_noisy,
    mopF_regime_switching,
)

MOP_CASES = [
    ("MOP-A — Monotonic redundancy", mopA_monotonic_redundancy),
    ("MOP-B — Trade-off + redundancies", mopB_tradeoff_with_redundancies),
    ("MOP-C — Latent blocks", mopC_latent_blocks_4x5),
    ("MOP-D — Pure conflict groups", mopD_pure_conflict_groups),
    ("MOP-E — Partial redundancy + noise", mopE_partial_redundancy_noisy),
    ("MOP-F — Regime switching", mopF_regime_switching),
]

# The original interactive benchmark used N=300. Set N=1000 for the canonical scientific sample size.
N = 300
SEED = 123

## Case runner

The helper below deliberately keeps the cases visible. It does not serialize or aggregate them: every `MISDAResult` is retained for subsequent inspection.

In [ ]:
def run_cases(cases, n=N, seed=SEED):
    results = {}

    for name, generator in cases:
        data, truth = generator(N=n, seed=seed)
        problem = truth.get("feature", truth.get("notes", ""))
        intuition = truth.get("intuition", "")
        graph_expected = truth.get("graph_expected", "")
        latent = truth.get("latent_expected", truth.get("intrinsic_dim_expected"))
        structural = truth.get("structural_expected", latent)

        print(f"\n{'=' * 80}")
        print(f"Running:    {name}")
        if problem:
            print(f"Problem:    {problem}")
        if intuition:
            print(f"Intuition:  {intuition}")
        if latent is not None:
            print(f"Latent:     {latent}")
        if structural is not None:
            print(f"Structural: {structural}")
        if graph_expected:
            print(f"Graph:      {graph_expected}")
        print('=' * 80)

        result = misda.analyze(
            data,
            method="static",
            name=name,
            seed=seed,
            max_evaluated_mis=1,
        )

        print(result.report())
        result.graph_plot()

        results[name] = {"result_obj": result, "truth": truth}

    return results

## Canonical structural cases

In [ ]:
canonical_results = run_cases(CANONICAL_CASES, n=N, seed=SEED)

## Synthetic multi-objective problems

In [ ]:
mop_results = run_cases(MOP_CASES, n=N, seed=SEED)